In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("dark_background")

import networkx as nx
from sklearn.manifold import SpectralEmbedding

def generate_community_graph(
    n_nodes: int,
    n_communities: int,
    p_intra: float,
    p_inter: float,
    seed: int = None
) -> nx.Graph:
    if seed is not None:
        np.random.seed(seed)

    sizes = [n_nodes // n_communities] * n_communities
    remainder = n_nodes % n_communities
    for i in range(remainder):
        sizes[i] += 1

    probs = np.full((n_communities, n_communities), p_inter)
    np.fill_diagonal(probs, p_intra)

    G = nx.stochastic_block_model(sizes, probs, seed=seed)

    community_labels = []
    for comm_id, size in enumerate(sizes):
        community_labels.extend([comm_id] * size)

    return G, community_labels

# -------------------------
# Build graph
# -------------------------
G, labels = generate_community_graph(
    n_nodes=20,
    n_communities=3,
    p_intra=0.8,
    p_inter=0.05,
    seed=42
)
assert nx.is_connected(G)

# -------------------------
# Adjacency matrix
# -------------------------
A = nx.to_numpy_array(G)

# -------------------------
# Robust community colors (used for BOTH graph + embedding)
# -------------------------
labels = np.asarray(labels)
communities = np.unique(labels)
n_comms = len(communities)

cmap = plt.get_cmap("jet")  # your choice
colors = [cmap(i / max(1, n_comms - 1)) for i in range(n_comms)]
comm_to_color = {c: colors[i] for i, c in enumerate(communities)}

# per-node colors aligned with node ordering 0..n-1
node_colors = [comm_to_color[labels[i]] for i in range(len(labels))]

# -------------------------
# Graph plot (colored by community like scatter)
# -------------------------
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(10,10))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, edgecolors="white", node_size=400)
nx.draw_networkx_edges(G, pos, alpha=0.6, edge_color="#90e0ef")
nx.draw_networkx_labels(G, pos, {i:f"{i}" for i in G}, font_color="white")
plt.title("Agent Graph (colored by community)", color="white")
plt.axis("off")
plt.show()

# -------------------------
# Adjacency heatmap
# -------------------------
plt.figure(figsize=(6,6))
sns.heatmap(A, square=True)
plt.title("Adjacency Matrix", color="white")
plt.show()

# -------------------------
# Spectral embedding (split into two lines)
# -------------------------
embedder = SpectralEmbedding(n_components=2, affinity="precomputed", random_state=42)
X = embedder.fit_transform(A)

plt.figure(figsize=(7,7))

# draw edges first so nodes appear on top
for i, j in G.edges():
    plt.plot(
        [X[i,0], X[j,0]],
        [X[i,1], X[j,1]],
        color="white",
        alpha=0.4,
        linewidth=1
    )

# draw nodes
for c in communities:
    mask = labels == c
    plt.scatter(
        X[mask, 0], X[mask, 1],
        s=160,
        edgecolors="white",
        linewidths=0.8,
        label=f"Community {c}",
        color=comm_to_color[c]
    )

plt.title("Spectral Embedding (colored by community)", color="white")
plt.xlabel("Component 1", color="white")
plt.ylabel("Component 2", color="white")
plt.legend(frameon=False)
plt.grid(alpha=0.15)
plt.show()

